# 07: FLOWFarm YAML Layout Optimization

In this example, we demonstrate a YAML-driven FLOWFarm setup in `Ard`, then run both a one-shot analysis and a layout optimization.

We start by importing the tools used throughout the notebook.

## Note
This example is the same as 01_onshore but uses FLOWFarm as the wind farm model.

## Inputs used

- `inputs/ard_system.yaml`
- `inputs/windio.yaml`

Now we set up the case from YAML.

The file `inputs/ard_system.yaml` contains both the Ard system definition and analysis options, and it references `inputs/windio.yaml` for wind plant data.

In [3]:
from pathlib import Path  # optional, for nice path specifications
import os
import sys
import importlib.util

import pprint as pp  # optional, for nice printing
import numpy as np  # numerics library
import matplotlib.pyplot as plt  # plotting capabilities

# Ensure the local Ard package is importable when running from this example folder.
repo_root = Path.cwd().resolve()
while repo_root.name != "Ard" and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Let Ard own JuliaCall bootstrap behavior.
os.environ.pop("PYTHON_JULIACALL_EXE", None)
os.environ.pop("PYTHON_JULIACALL_PROJECT", None)

import ard  # package import
from ard.utils.io import load_yaml  # yaml loader
from ard.api import set_up_ard_model  # model setup
from ard.viz.layout import plot_layout  # layout plotting helper

import openmdao.api as om  # for optional N2 diagrams

%matplotlib inline

In [4]:
# load input
path_inputs = Path.cwd().absolute() / "inputs"
input_dict = load_yaml(path_inputs / "ard_system.yaml")

# create and setup system
prob = set_up_ard_model(input_dict=input_dict, root_data_path=path_inputs)
prob

Running OpenMDAO util to clean the output directories...
	Found 1 OpenMDAO output directories:
	Removed case_files/flowfarm_yaml_setup_demo_out
	Removed 1 OpenMDAO output directories.
... done.

Created top-level OpenMDAO problem: top_level.
Adding top_level.
	Adding layout2aep.
		Adding layout to layout2aep.
		Adding aepFLOWFarm to layout2aep.
	Adding boundary.
	Adding spacing_constraint.
	Adding landuse.
	Adding collection.
	Adding tcc.
	Adding landbosse.
	Adding opex.
	Adding financese.
System top_level built.
System top_level set up.


/opt/homebrew/Caskroom/miniconda/base/envs/ard-env/lib/python3.12/site-packages/openmdao/core/problem.py:351: OpenMDAOWarning:The problem name 'flowfarm_yaml_setup_demo' already exists
/opt/homebrew/Caskroom/miniconda/base/envs/ard-env/lib/python3.12/site-packages/openmdao/utils/reports_system.py:277: OpenMDAOWarning:A report with the name 'scaling' for instance 'flowfarm_yaml_setup_demo.driver' is already active.
/opt/homebrew/Caskroom/miniconda/base/envs/ard-env/lib/python3.12/site-packages/openmdao/utils/reports_system.py:277: OpenMDAOWarning:A report with the name 'total_coloring' for instance 'flowfarm_yaml_setup_demo.driver' is already active.
/opt/homebrew/Caskroom/miniconda/base/envs/ard-env/lib/python3.12/site-packages/openmdao/utils/reports_system.py:277: OpenMDAOWarning:A report with the name 'n2' for instance 'flowfarm_yaml_setup_demo' is already active.
/opt/homebrew/Caskroom/miniconda/base/envs/ard-env/lib/python3.12/site-packages/openmdao/utils/reports_system.py:277: Ope

You can optionally inspect the OpenMDAO N2 diagram for debugging and model introspection. It is left off by default.

In [ ]:
if False:
    # visualize model
    om.n2(prob)

In [ ]:
# run the model
prob.run_model()

# collapse the test result data
test_data = {
        "AEP_val": float(prob.get_val("AEP_farm", units="GW*h")[0]),
        "CapEx_val": float(prob.get_val("tcc.tcc", units="MUSD")[0]),
        "BOS_val": float(prob.get_val("landbosse.total_capex", units="MUSD")[0]),
        "OpEx_val": float(prob.get_val("opex.opex", units="MUSD/yr")[0]),
        "LCOE_val": float(prob.get_val("financese.lcoe", units="USD/MW/h")[0]),
        "area_tight": float(prob.get_val("landuse.area_tight", units="km**2")[0]),
        "coll_length": float(
            prob.get_val("collection.total_length_cables", units="km")[0]
        ),
        "turbine_spacing": float(
            np.min(prob.get_val("spacing_constraint.turbine_spacing", units="km"))
        ),
    }

print("\n\nRESULTS:\n")
pp.pprint(test_data)
print("\n\n")

Now we can optimize the same problem.

Optimization settings are defined in `inputs/ard_system.yaml` under `analysis_options`, where `spacing_primary`, `spacing_secondary`, `angle_orientation`, and `angle_skew` are design variables.

For parity with `01_onshore`, this setup includes the cost/finance chain and optimizes `financese.lcoe` (minimized). Boundary and spacing constraints are enforced through `boundary_distances` and `spacing_constraint.turbine_spacing`.

In [ ]:
optimize = True  # set to False to skip optimization
if optimize:
    # run the optimization
    prob.run_driver()
    prob.cleanup()

    # collapse the test result data
    test_data = {
        "AEP_val": float(prob.get_val("AEP_farm", units="GW*h")[0]),
        "CapEx_val": float(prob.get_val("tcc.tcc", units="MUSD")[0]),
        "BOS_val": float(prob.get_val("landbosse.total_capex", units="MUSD")[0]),
        "OpEx_val": float(prob.get_val("opex.opex", units="MUSD/yr")[0]),
        "LCOE_val": float(prob.get_val("financese.lcoe", units="USD/MW/h")[0]),
        "area_tight": float(prob.get_val("landuse.area_tight", units="km**2")[0]),
        "coll_length": float(
            prob.get_val("collection.total_length_cables", units="km")[0]
        ),
        "turbine_spacing": float(
            np.min(prob.get_val("spacing_constraint.turbine_spacing", units="km"))
        ),
    }

    # clean up the recorder
    prob.cleanup()

    # print the results
    print("\n\nRESULTS (opt):\n")
    pp.pprint(test_data)
    print("\n\n")

In [ ]:
plot_layout(
    prob,
    input_dict=input_dict,
    show_image=True,
    include_cable_routing=True,
)
plt.show()


In [ ]:
prob.cleanup()